## 1. Default Message

In [1]:
# Without helper

from gai.messages.typing import MessagePydantic, DefaultBodyPydantic
import time
message = MessagePydantic(
    **{
        "header": {
            "sender": "User",
            "recipient": "Assistant",
            "timestamp": time.time(),
        },
        "body": {
            "type": "default",
            "content": "Hello, how are you?",
        }
    }
)
print(message)
assert isinstance(message.body,DefaultBodyPydantic)

# With helper

from gai.messages import create_message, convert_to_chat_messages

user_message = create_message(role="user", content="What is the capital of France?")
print(user_message)
assistant_message = create_message(role="assistant", content="The capital of France is Paris.")
print(assistant_message)

assert user_message.header.sender == "User"
assert user_message.header.recipient == "Assistant"
assert user_message.body.content == "What is the capital of France?"

assert assistant_message.header.sender == "Assistant"
assert assistant_message.header.recipient == "User"
assert assistant_message.body.content == "The capital of France is Paris."

gai_messages = [user_message, assistant_message]
convert_to_chat_messages(gai_messages)


id='e19e0043-abb5-49d6-98fb-39f9f974f568' header=MessageHeaderPydantic(sender='User', recipient='Assistant', timestamp=1752306529.2780032, order=0) body=DefaultBodyPydantic(type='default', content='Hello, how are you?')
id='8d65ae8a-7345-4154-aa8d-7bb25396b645' header=MessageHeaderPydantic(sender='User', recipient='Assistant', timestamp=1752306529.2790828, order=0) body=DefaultBodyPydantic(type='default', content='What is the capital of France?')
id='334a36be-1621-410f-bbff-8f6f16824a61' header=MessageHeaderPydantic(sender='Assistant', recipient='User', timestamp=1752306529.2793367, order=0) body=DefaultBodyPydantic(type='default', content='The capital of France is Paris.')


[]

## 2. SendMessagePydantic vs ReplyMessagePydantic


In [2]:
from gai.messages import create_user_send_message,create_assistant_reply_chunk, convert_to_chat_messages
user_message = create_user_send_message(
    recipient="Sara",
    content="Tell me a story about a brave knight.",
)
print(user_message)
assistant_message = create_assistant_reply_chunk(sender="Sara",recipient="User", chunk_no=0, chunk="<eom>", content="Once upon a time, a brave knight saved a kingdom from a dragon.")
print(assistant_message)

assert user_message.header.sender == "User"
assert user_message.header.recipient == "Sara"
assert user_message.body.content == "Tell me a story about a brave knight."

assert assistant_message.header.sender == "Sara"
assert assistant_message.header.recipient == "User"
assert assistant_message.body.content == "Once upon a time, a brave knight saved a kingdom from a dragon."

gai_messages = [user_message, assistant_message]
convert_to_chat_messages(gai_messages)



id='6ff30c62-9043-4f5d-9282-f19809cbd29b' header=MessageHeaderPydantic(sender='User', recipient='Sara', timestamp=1752306532.0018914, order=0) body=SendBodyPydantic(type='send', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=4, message_id='00000000-0000-0000-0000-000000000000.4', content_type='text', content='Tell me a story about a brave knight.')
id='f0f40541-e0e4-4b99-8ad5-57404ac1433a' header=MessageHeaderPydantic(sender='Sara', recipient='User', timestamp=1752306532.0023437, order=0) body=ReplyBodyPydantic(type='reply', dialogue_id='00000000-0000-0000-0000-000000000000', message_no=5, message_id='00000000-0000-0000-0000-000000000000.5', chunk_no=0, chunk='<eom>', content_type='text', content='Once upon a time, a brave knight saved a kingdom from a dragon.')


[{'role': 'user', 'content': 'Tell me a story about a brave knight.'},
 {'role': 'assistant',
  'content': 'Once upon a time, a brave knight saved a kingdom from a dragon.'}]

## 3. Serialization and Deserialization

In [3]:
from gai.messages import create_user_send_message,create_assistant_reply_chunk, convert_to_chat_messages,json,unjson
from gai.messages.typing import ReplyBodyPydantic, SendBodyPydantic
user_message = create_user_send_message(
    recipient="Sara",
    content="Tell me a story about a brave knight.",
)
assistant_message = create_assistant_reply_chunk(sender="Sara",recipient="User", chunk_no=0, chunk="<eom>", content="Once upon a time, a brave knight saved a kingdom from a dragon.")
gai_messages = [user_message, assistant_message]

# Convert array to JSON
jsoned = json(gai_messages)
print(jsoned)

# Convert JSON back as array and should not lose resolution
gai_messages_again = unjson(jsoned)
print(gai_messages_again)
assert gai_messages_again[0].header.sender == "User"
assert gai_messages_again[0].header.recipient == "Sara"
assert isinstance(gai_messages_again[0].body, SendBodyPydantic)

assert gai_messages_again[1].header.sender == "Sara"
assert gai_messages_again[1].header.recipient == "User"
assert isinstance(gai_messages_again[1].body, ReplyBodyPydantic)



[
    {
        "id": "2c18e19d-174f-4835-aff0-9b7f5623fc66",
        "header": {
            "sender": "User",
            "recipient": "Sara",
            "timestamp": 1752306533.9691625,
            "order": 0
        },
        "body": {
            "type": "send",
            "dialogue_id": "00000000-0000-0000-0000-000000000000",
            "message_no": 6,
            "message_id": "00000000-0000-0000-0000-000000000000.6",
            "content_type": "text",
            "content": "Tell me a story about a brave knight."
        }
    },
    {
        "id": "4a264bf7-66c4-477d-9544-8ce798144215",
        "header": {
            "sender": "Sara",
            "recipient": "User",
            "timestamp": 1752306533.969315,
            "order": 0
        },
        "body": {
            "type": "reply",
            "dialogue_id": "00000000-0000-0000-0000-000000000000",
            "message_no": 7,
            "message_id": "00000000-0000-0000-0000-000000000000.7",
            "chunk

---

## 2. Async Message Bus

### a) Quick Start - Publish to LLM

In [ ]:
from gai.messages import (
    MessagePydantic, 
    AsyncMessageBus, 
    create_user_send_message,
    create_assistant_reply_chunk
)    
from rich import print

amb = AsyncMessageBus()

# 1. Start the background bus loop

await amb.start()
print("Bus is ready.")

# 2. Subscribe and handle send to LLM

async def handle_send(msg: MessagePydantic):
    
    # Forward user message to LLM
    
    from gai.llm.openai import OpenAI
    client = OpenAI(client_config={"model":"gpt-4o-mini"})
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user","content":msg.body.content}],
        max_tokens=100
        )
    
    # Forward LLM response to user
    
    await amb.publish(
        create_assistant_reply_chunk(
            sender=msg.header.recipient,  
            recipient=msg.header.sender,
            chunk_no=0,
            chunk="<eom>",
            content=resp.choices[0].message.content,
        )
    )
await amb.subscribe("send", {"Alice": handle_send})
    
# 3. Subscribe and handle replies    

async def handle_reply(msg: MessagePydantic):
    print(f"Reply from {msg.header.sender}: [green]{msg.body.content}[/]")
await amb.subscribe("reply", {"User": handle_reply})

# 4. Publish a message to the bus

msg = create_user_send_message(content="Hello, what is your name?")
await amb.publish(msg)


Bus is ready.

Reply from Assistant: Hello! I'm an AI language model, and I don't have a personal name, but you can call me 
Assistant. How can I help you today?


### b) Same Message Type - Multiple Subscribers

In [5]:

import asyncio
from gai.messages import MessagePydantic, AsyncMessageBus, create_user_send_message
from rich import print

amb = AsyncMessageBus()

# 1. Start the background bus loop
await amb.start()
print("Bus is ready.")

# 2. Create 2 handlers to show multiple subscriptions to the same message type

async def handler_1(msg: MessagePydantic):
    print(f"[bright_yellow] handler_1 > {msg}[/bright_yellow]")

async def handler_2(msg: MessagePydantic):
    print(f"[bright_green] handler_2 > {msg}[/bright_green]")

# 2. Subscribe the callback to the message type="send"

await amb.subscribe("send", {"Alice": handler_1})
await amb.subscribe("send", {"Bob":handler_2})
print("Subscribed to message type 'send'")

# 4. Confirm the bus is started and publish a message

if not amb.is_started:
    raise RuntimeError("Bus is not started, cannot publish message.")

msg = create_user_send_message(
    recipient="Assistant",
    content="Hello, what is your name?"
    )

print("Publishing message.")
await amb.publish(msg)
print("Message published.")
await asyncio.sleep(0.1)  # Give time for the message to be processed

# 5. Wait a little for message to be received before cancelling task
await amb.stop()

print("Bus stopped.")


Bus is ready.

Subscribed to message type 'send'

Publishing message.

Message published.

 handler_1 > id='9b6a5afa-a4e1-4c29-8c51-a1a2c8655447' header=MessageHeaderPydantic(sender='User', 
recipient='Assistant', timestamp=1752306540.1093714, order=0) body=SendBodyPydantic(type='send', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=12, 
message_id='00000000-0000-0000-0000-000000000000.12', content_type='text', content='Hello, what is your name?')

 handler_2 > id='9b6a5afa-a4e1-4c29-8c51-a1a2c8655447' header=MessageHeaderPydantic(sender='User', 
recipient='Assistant', timestamp=1752306540.1093714, order=0) body=SendBodyPydantic(type='send', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=12, 
message_id='00000000-0000-0000-0000-000000000000.12', content_type='text', content='Hello, what is your name?')

Bus stopped.

### b) Multiple Message Type - One Subscriber

In [6]:
import asyncio
from gai.messages import (
    MessagePydantic, 
    AsyncMessageBus, 
    create_user_send_message,
    create_assistant_reply_chunk
    )
from rich import print

amb = AsyncMessageBus()

# 1. Start the background bus loop AND wait for signal to be ready
await amb.start()
print("Bus is ready.")

# 2. Create 2 handlers: one for SendMessagePydantic and one for ReplyMessagePydantic

async def send_handler(msg: MessagePydantic):
    print(f"[bold bright_yellow]Send Message Received: {msg}[/bold bright_yellow]")

async def reply_handler(msg: MessagePydantic):
    print(f"[bold bright_green]Send Message Received: {msg}[/bold bright_green]")

# 3. Subscribe the callback to the message type="alpha"

await amb.subscribe("send", {"Alice":send_handler})
await amb.subscribe("reply", {"Alice":reply_handler})
print("Subscribed to send and reply message types")

# 4. Confirm the bus is started and publish a message

if not amb.is_started:
    raise RuntimeError("Bus is not started, cannot publish message.")

send_msg = create_user_send_message(content="Tell me a one sentence story.")

print("Publishing send message.")
await amb.publish(send_msg)
print("Send message published.")

reply_msg = create_assistant_reply_chunk(sender="Sara",chunk_no=0, chunk="<eom>", content="Once upon a time, a brave knight saved a kingdom from a dragon.")
print("Publishing reply message.")
await amb.publish(reply_msg)
print("Reply message published.")

# 5. Wait a little for message to be received before cancelling task

await asyncio.sleep(0.1)

# 6. Stop the bus
await amb.stop()
print("Bus stopped.")


Bus is ready.

Subscribed to send and reply message types

Publishing send message.

Send message published.

Publishing reply message.

Reply message published.

Send Message Received: id='1dba6bdd-aeb2-42bf-bc3a-6fc1f33a8656' header=MessageHeaderPydantic(sender='User', 
recipient='Assistant', timestamp=1752306542.6250446, order=0) body=SendBodyPydantic(type='send', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=13, 
message_id='00000000-0000-0000-0000-000000000000.13', content_type='text', content='Tell me a one sentence story.')

Send Message Received: id='e3246520-be65-4048-be9a-920ccd80d909' header=MessageHeaderPydantic(sender='Sara', 
recipient='User', timestamp=1752306542.6304152, order=0) body=ReplyBodyPydantic(type='reply', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=14, 
message_id='00000000-0000-0000-0000-000000000000.14', chunk_no=0, chunk='<eom>', content_type='text', content='Once
upon a time, a brave knight saved a kingdom from a dragon.')

Bus stopped.

### c) Same Message Type - Same Handler - Different Handler State

In [7]:
import asyncio
from gai.messages import (
    MessagePydantic,
    AsyncMessageBus,
    create_assistant_reply_content,
    create_user_send_message
)
    
from rich import print

# 3. Start the background bus loop AND wait for signal to be ready

amb = AsyncMessageBus()
await amb.start()
print("Bus is ready.")

# 1. Create 2 assistant send_handlers to show multiple subscriptions to the same message type

class Host:
    def __init__(self,name:str):
        self.name = name
        
    async def send_handler(self, msg: MessagePydantic):
        print(f"[bold bright_yellow]{self.name} > {msg}[/bold bright_yellow]")
        await amb.publish(
            create_assistant_reply_content(
                sender=self.name,
                recipient="User",
                content=f"My name is {self.name}."                            
            )
        )
        
host_1 = Host("Host 1")
host_2 = Host("Host 2")

# 2. Create 1 user reply_handlers to handle replies

async def reply_handler(msg: MessagePydantic):
    print(f"[bold bright_green]User > {msg}[/bold bright_green]")


# 2. Subscribe the callback to the message type="alpha" but same handler belonging to different objects

await amb.subscribe("send", {"Alice":host_1.send_handler})
await amb.subscribe("send", {"Bob":host_2.send_handler})
await amb.subscribe("reply", {"User":reply_handler})

print("Subscribed to message type 'send'")

# 4. Confirm the bus is started and publish a message

if not amb.is_started:
    raise RuntimeError("Bus is not started, cannot publish message.")

msg = create_user_send_message(
    recipient="*",
    content="Hello, what is your name?"
)
print("Publishing message.")
await amb.publish(msg)
print("Message published.")

# 5. Wait a little for message to be received before cancelling task

await asyncio.sleep(0.1)
status=await amb.stop()
print("Bus stopped.")


Bus is ready.

Subscribed to message type 'send'

Publishing message.

Message published.

Task was destroyed but it is pending!
task: <Task pending name='Task-6' coro=<AsyncMessageBus._dispatch_loop() running at /workspaces/gai-mace/src/gai/messages/async_message_bus.py:313> wait_for=<Future pending cb=[Task.task_wakeup()]>>


Host 1 > id='96552352-7da1-47c0-b39d-56c5faef2e08' header=MessageHeaderPydantic(sender='User', recipient='*', 
timestamp=1752306548.225728, order=0) body=SendBodyPydantic(type='send', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=15, 
message_id='00000000-0000-0000-0000-000000000000.15', content_type='text', content='Hello, what is your name?')

Host 2 > id='96552352-7da1-47c0-b39d-56c5faef2e08' header=MessageHeaderPydantic(sender='User', recipient='*', 
timestamp=1752306548.225728, order=0) body=SendBodyPydantic(type='send', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=15, 
message_id='00000000-0000-0000-0000-000000000000.15', content_type='text', content='Hello, what is your name?')

User > id='e33637c7-2442-4326-b38a-8967f36a93a7' header=MessageHeaderPydantic(sender='Host 1', recipient='User', 
timestamp=1752306548.3167403, order=0) body=ReplyBodyPydantic(type='reply', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=16, 
message_id='00000000-0000-0000-0000-000000000000.16', chunk_no=0, chunk='<eom>', content_type='text', content='My 
name is Host 1.')

User > id='e14f355d-29a8-451e-8209-20f5f13900ea' header=MessageHeaderPydantic(sender='Host 2', recipient='User', 
timestamp=1752306548.3194137, order=0) body=ReplyBodyPydantic(type='reply', 
dialogue_id='00000000-0000-0000-0000-000000000000', message_no=17, 
message_id='00000000-0000-0000-0000-000000000000.17', chunk_no=0, chunk='<eom>', content_type='text', content='My 
name is Host 2.')

Bus stopped.